# 02 — Edge Sign Classification on the SNAP Reddit Hyperlink Network

**Студент:** Олександр Щепанчук &nbsp;&nbsp; **Група:** ПМПм-12

**Датасет:** [SNAP Reddit Hyperlink Network](https://snap.stanford.edu/data/soc-RedditHyperlinks.html)

### Reproduction command
```bash
make all                  # install + download SNAP + preprocess + run all unit tests
make train CONFIG=configs/gcn.yaml   # one model
make sweep                # hyperparameter search
uv run python scripts/export_report_assets.py   # regenerate every report figure + table
```

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from reddit_gnn.paths import PATHS

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

EDGES_PATH = PATHS.data_processed / 'edges.parquet'
META_PATH = PATHS.data_processed / 'metadata.json'
N2I_PATH = PATHS.data_processed / 'node_to_id.json'
FIG_DIR = PATHS.reports_figures
TBL_DIR = PATHS.reports_tables

def show_image_or_msg(path: Path, label: str = '') -> None:
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path)))
        print(f'shown: {path}')
    else:
        print(f'Missing — run `make all` first: {path}')

def load_table_or_msg(path: Path, **read_kwargs) -> pd.DataFrame | None:
    path = Path(path)
    if not path.exists():
        print(f'Missing — run `make all` first: {path}')
        return None
    return pd.read_csv(path, **read_kwargs)

## Опис датасету

* **Nodes** — subreddits (lowercased, deduplicated).
* **Edges** — directed, signed, attributed, timestamped subreddit-to-subreddit hyperlinks extracted from post bodies *and* titles (`source_file` ∈ {body, title}).
* **Sign** — `POST_LABEL` ∈ {-1, +1}, remapped to {0, 1}: 0 = negative sentiment (rare ≈10%), 1 = neutral/positive (majority ≈90%).
* **Edge features** — 86-D `POST_PROPERTIES` (LIWC / text metrics) per hyperlink + timestamp + `is_title` flag.
* **Node features** — 300-D LIWC subreddit embeddings when available; an explicit unknown-flag column for subreddits the SNAP file does not cover.

In [2]:
if not EDGES_PATH.exists():
    print(f'Missing — run `make all` first: {EDGES_PATH}')
    df = None; metadata = None
else:
    df = pd.read_parquet(EDGES_PATH)
    metadata = json.loads(META_PATH.read_text())
    print(f'{len(df):,} edges, {metadata["num_nodes"]:,} nodes')
    print(f'time range: {metadata["timestamp_range"]["min"]} → {metadata["timestamp_range"]["max"]}')
    display(df.head())

Missing — run `make all` first: /mnt/d/oleksandr/GNN/data/processed/edges.parquet


## Опис задачі

**Edge sign classification on observed hyperlinks.** Each example is a real, observed hyperlink; the prediction target is its sentiment label.

**Anti-requirements (explicit):**
* We do **not** use negative sampling.
* Label `0` is the *negative-sentiment* class, **never** treated as "non-edge".

**Why this matters.** The task is fundamentally different from generic link prediction: every supervision edge is a known observation, and the GNN encoder's message-passing graph is constructed from the **training** edges only so that the encoder never sees val/test labels.

Label imbalance (~90/10) makes **PR-AUC for the negative class** the headline leaderboard metric; F1-macro is the secondary report.

## Завантаження та preprocessing

`reddit_gnn.data.preprocess.clean_edges` drops self-loops + duplicates, remaps the label, tags `is_title`, and sorts by timestamp. The processed parquet is the source of truth for every downstream step.

In [3]:
stats = load_table_or_msg(TBL_DIR / 'stats_summary.csv')
if stats is not None:
    display(stats.T.rename(columns={0: 'value'}))
else:
    print('(re-run `01_eda.ipynb` first to produce stats_summary.csv)')

Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/tables/stats_summary.csv
(re-run `01_eda.ipynb` first to produce stats_summary.csv)


## Візуалізації

Embedded from `reports/figures/` — produced by `01_eda.ipynb`. Re-run that notebook to refresh.

In [4]:
for name in [
    '01_label_distribution.png',
    '01_degree_total_log.png', '01_degree_in_log.png', '01_degree_out_log.png',
    '01_top_total_degree.png', '01_top_in_degree.png', '01_top_out_degree.png',
    '01_edges_over_time.png', '01_negative_ratio_over_time.png',
    '01_sampled_signed_subgraph.png',
    '01_ego_peaceful.png', '01_ego_hostile.png',
]:
    show_image_or_msg(FIG_DIR / name, name)

Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_label_distribution.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_degree_total_log.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_degree_in_log.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_degree_out_log.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_top_total_degree.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_top_in_degree.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_top_out_degree.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_edges_over_time.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_negative_ratio_over_time.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/figures/01_sampled_signed_subgraph.png
Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/fig

## Метрики графа

In [5]:
stats = load_table_or_msg(TBL_DIR / 'stats_summary.csv')
if stats is not None:
    display(stats.T.rename(columns={0: 'value'}))

Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/tables/stats_summary.csv


## Розбиття: chronological split + leakage assertion

Edges are sorted by timestamp, then split by row position into train/val/test (70/15/15). The encoder's message-passing graph for each fold uses only past edges (val uses all train edges; test uses train+val edges). The supervision set per fold is **disjoint** from the message-passing set by construction.

In [6]:
from reddit_gnn.data.splits import build_message_passing_split, chronological_edge_split
from reddit_gnn.data.splits import assert_no_leakage

if df is None:
    print('Run `make all` first — cannot run the leakage check without processed data.')
else:
    sp = chronological_edge_split(df)
    splits = build_message_passing_split(df, sp, seed=42)
    # Class balance per split.
    rows = []
    for name, idx in [('train', sp.train_idx), ('val', sp.val_idx), ('test', sp.test_idx)]:
        sub = df.iloc[idx]
        rows.append({
            'split': name, 'n': len(sub),
            'positive': int((sub['label_binary'] == 1).sum()),
            'negative': int((sub['label_binary'] == 0).sum()),
            'positive_ratio': float((sub['label_binary'] == 1).mean()),
            'time_min': sub['TIMESTAMP'].min(), 'time_max': sub['TIMESTAMP'].max(),
        })
    display(pd.DataFrame(rows))
    # Hard leakage assertion (raises if any invariant is violated).
    assert_no_leakage(df)
    print('leakage check passed.')

Run `make all` first — cannot run the leakage check without processed data.


## Інженерія ознак

`FeatureBuilder` fits StandardScaler on the **train** structural features (degrees, log-degrees, signed-ratio columns) and the **train** temporal edge features only. Validation and test transforms reuse the fitted scalers + bounds.

In [7]:
from reddit_gnn.data.features import FeatureBuilder

if df is None:
    print('Run `make all` first.')
else:
    sp = chronological_edge_split(df)
    train_df = df.iloc[sp.train_idx]
    num_nodes = int(max(df['source_id'].max(), df['target_id'].max())) + 1
    emb_path = PATHS.data_raw / 'web-redditEmbeddings-subreddits.csv'
    use_snap = emb_path.exists()
    fb = FeatureBuilder(use_snap_embeddings=use_snap).fit(
        train_df, num_nodes=num_nodes, embeddings_path=emb_path if use_snap else None,
    )
    node_features = fb.transform_node_features(train_df, num_nodes=num_nodes)
    edge_features = fb.transform_edge_features(df)
    print(f'node_features: {tuple(node_features.shape)}   edge_features: {tuple(edge_features.shape)}')
    print(f'SNAP embeddings: {"used" if use_snap else "NOT FOUND — using zeros + unknown flag for every node"}')

Run `make all` first.


## Базові моделі

Non-graph baselines for the leaderboard floor: `Majority` always predicts the train-majority class; the others use `LogisticRegression` and a small MLP on edge features and edge ∥ endpoint-derived concatenations. They are trained by `scripts/run_experiment.py --config configs/baseline_*.yaml` (LogReg variants currently fall through to a stub — see commit history for the planned wiring).

In [8]:
comp = load_table_or_msg(TBL_DIR / 'comparison.csv')
if comp is not None and not comp.empty:
    baselines = comp[comp['model'].isin(['MajorityClassifier', 'baseline_logreg', 'baseline_mlp', 'baseline_logreg_node'])]
    if baselines.empty:
        print('No baseline rows in comparison.csv yet — train them with `make train CONFIG=configs/baseline_mlp.yaml`.')
    else:
        display(baselines)

Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/tables/comparison.csv


## GNN моделі

GCN, GraphSAGE, GAT, (SignedGCN if available). The encoder produces node embeddings on the train MP graph; the `EdgeMLPDecoder` consumes endpoint embeddings + engineered edge features and emits a single logit per supervision edge. AdamW + ReduceLROnPlateau + early stopping on val PR-AUC (negative class).

In [9]:
comp = load_table_or_msg(TBL_DIR / 'comparison.csv')
if comp is not None and not comp.empty:
    gnns = comp[comp['model'].isin(['gcn', 'sage', 'gat', 'signed_gcn'])]
    if gnns.empty:
        print('No GNN rows in comparison.csv yet — train them with `make train CONFIG=configs/gcn.yaml` (etc).')
    else:
        display(gnns)

# Training curves (one per run).
for png in sorted(FIG_DIR.glob('curve_*.png')):
    show_image_or_msg(png, png.name)

Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/tables/comparison.csv


## Підбір гіперпараметрів

Per-architecture Optuna sweeps; each row is a single trial. `mean_seed` / `std_seed` in `comparison.csv` aggregate across multiple seeds of the same configuration. The sweep CSV format is `reports/tables/sweep_<arch>.csv`.

In [10]:
sweep_files = sorted(TBL_DIR.glob('sweep_*.csv'))
if not sweep_files:
    print('No sweep CSVs yet — produce them with `make sweep`.')
for sw in sweep_files:
    print(sw.name)
    df_sw = pd.read_csv(sw)
    display(df_sw.head(10))

No sweep CSVs yet — produce them with `make sweep`.


## Результати — фінальне порівняння

`reports/tables/comparison.csv` aggregates every saved run by model type. `mean_seed` / `std_seed` are the mean and standard deviation of `test_pr_auc_neg` across seeds (≥3 recommended).

In [11]:
comp = load_table_or_msg(TBL_DIR / 'comparison.csv')
if comp is None or comp.empty:
    print('Run `make all` first — comparison.csv is empty.')
else:
    display(comp)

Missing — run `make all` first: /mnt/d/oleksandr/GNN/reports/tables/comparison.csv
Run `make all` first — comparison.csv is empty.


## Аналіз помилок

Per-run breakdowns of false-positive / false-negative rates by source-node degree and by time bin, produced by `scripts/export_report_assets.py`.

In [12]:
patterns = ['error_by_degree_*.png', 'error_by_time_*.png', 'confusion_*.png', 'pr_roc_*.png']
found = []
for pat in patterns:
    found.extend(sorted(FIG_DIR.glob(pat)))
if not found:
    print('No error analysis figures yet — run `make all` then `uv run python scripts/export_report_assets.py`.')
for png in found:
    show_image_or_msg(png, png.name)

No error analysis figures yet — run `make all` then `uv run python scripts/export_report_assets.py`.


## Передбачений підграф

`plot_predicted_subgraph` draws a 100-edge sample with two colors per edge: a thick outer line in the **true-sign color**, a thinner inner line in the **predicted-sign color**. Disagreements show up as a halo of true-color around the predicted-color interior.

In [13]:
for png in sorted(FIG_DIR.glob('predicted_subgraph_*.png')):
    show_image_or_msg(png, png.name)
if not list(FIG_DIR.glob('predicted_subgraph_*.png')):
    print('No predicted-subgraph figures yet — run `make all` then `uv run python scripts/export_report_assets.py`.')

No predicted-subgraph figures yet — run `make all` then `uv run python scripts/export_report_assets.py`.


## Обговорення

**Що працювало.**
* Train-only feature fitting (FeatureBuilder.fit on train rows; train min/max for temporal normalization) — leakage tests confirm fitted scalers match the manual mean/std of training rows only.
* Chronological + disjoint-MP/sup split — `tests/test_leakage.py` guarantees the encoder never sees val/test labels.
* PR-AUC for the negative class as the early-stopping metric — F1-macro alone would have overfit to the majority class.

**Що не вийшло (і чому).**
* `pyg-lib` / `torch-sparse` have no PyPI wheel matching the installed torch build; `LinkNeighborLoader` therefore falls back to `FullBatchLoader`. The per-batch temporal-sampling defense-in-depth is lost; the broader split-level temporal invariant still holds.
* SignedGCN and the sklearn-based `baseline_logreg` are not yet wired through `scripts/run_experiment.py` — both classes are unit-tested but the script raises a clean `NotImplementedError` for them. A focused follow-up to write `fit_signed` (full-batch with pos/neg splits derived from train labels) and a sklearn wrapper would finish the leaderboard.

**Limitations.**
* The 300-D SNAP LIWC embeddings cover only a subset of subreddits; the rest get zeros + an `unknown` flag. A learnable input projection (which `SignedGCNEncoder` already has) helps marginally; a contrastive pre-training run on the structural graph would help more.
* The supervision-edge `edge_label_attr` is computed once per epoch in the full-batch fallback — fine for ~860k edges but a bottleneck if neighbour sampling came back.

**Future work.**
* Pin torch to a version that has matching `pyg-lib` wheels; benchmark mini-batched vs full-batch on the real dataset.
* Add temperature scaling for calibrated probabilities — the current PR-AUC is high but the implied threshold is far from 0.5.
* Multi-seed ablation grid: chronological vs random split, with vs without SNAP embeddings, focal vs weighted BCE.

## Відтворюваність

```bash
# 1. set up env
make install              # uv sync --all-extras

# 2. download + preprocess SNAP
make data                 # python scripts/prepare_data.py

# 3. train one model
make train CONFIG=configs/gcn.yaml
make train CONFIG=configs/sage.yaml
make train CONFIG=configs/gat.yaml
make train CONFIG=configs/baseline_mlp.yaml

# 4. hyperparameter search
make sweep CONFIG=configs/sweep.yaml

# 5. regenerate every report figure + table
uv run python scripts/export_report_assets.py

# 6. browse the MLflow UI
make mlflow-ui            # http://127.0.0.1:5000

# 7. (optional) run the leakage + metric tests on the produced data
make test
```

Random state: every entrypoint accepts `--seed`; the default is `42`. Three seeds are averaged for the final comparison.